# Introduction

// TODO

## Dataset

[CommonsenseQA](https://huggingface.co/datasets/tau/commonsense_qa) is a multiple-choice question answering dataset that requires common sense knowledge to select the correct answer from five choices.

# Setup

### Installations for GPU Hub

In [ ]:
%pip install transformers
%pip install datasets
%pip install wandb

## Imports

This section contains all the imports that are required in this notebook and gives a brief overview which packages are being used.

In [ ]:
import time
import random

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

from transformers import (
    RobertaConfig, 
    RobertaForMultipleChoice, 
    RobertaTokenizer,
    get_linear_schedule_with_warmup
)

from tqdm.auto import tqdm

from huggingface_hub import hf_hub_download
from datasets import load_dataset

import wandb


device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
if torch.backends.mps.is_available():
    device = torch.device("mps")
print(f'Using device for computations: {device}')


## Seed

Setting a fixed random seed ensures **reproducibility** of results across runs. We set the seed for NumPy and PyTorch random number generators to ensure that random operations like weight initialization and data shuffling produce the same results each time the notebook is executed.

In [ ]:
SEED = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.mps.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED) 

## Experiment Tracking

[Weights and Biases](https://wandb.ai/site/) is a machine learning experiment tracking and visualization platform that helps track, compare, and optimize models.

The dashboard is available here: // TODO: add link to view for project 2

In [ ]:
wandb.login()

The `initialize_wandb_run` function configures a new wandb run with metadata about the experiment.

In [ ]:
def initialize_wandb_run(
    run_name,
    learning_rate,
    batch_size,
    num_epochs,
    model_name,  # Added parameter for model architecture
    optimizer_name="AdamW",  # Added parameter for optimizer type
    weight_decay=0.01,  # Added parameter for weight decay
    warmup_ratio=0.1,  # Added parameter for warmup ratio
    max_seq_length=None,  # Added parameter for sequence length
    gradient_accumulation_steps=1,  # Added for gradient accumulation
    entity_name="timon-schmid-hochschule-luzern",
    project_name="hslu-nlp-commonsense_qa",
):
    
    wandb.init(
        entity=entity_name,
        project=project_name,
        name=run_name,
        config={
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "num_epochs": num_epochs,
            "model_name": model_name,  # Track model architecture
            "optimizer": optimizer_name,
            "weight_decay": weight_decay,
            "warmup_ratio": warmup_ratio,
            "max_seq_length": max_seq_length,
            "gradient_accumulation_steps": gradient_accumulation_steps,
            "gradient_clipping": 1.0,  # Common value for transformer training
            
            # Additional configuration details as per project requirements
            "dataset": "tau/commonsense_qa",
        }
    )
    
    # Return the wandb run object for further logging
    return wandb

# Preprocessing

## Tokenization

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

## Data Loading

### Data Split

We use the data split that was mention in the course lecture, because the test set does not contain the answer key.

In [ ]:
train_dataset = load_dataset("tau/commonsense_qa", split="train[:-1000]")
validation_dataset = load_dataset("tau/commonsense_qa", split="train[-1000:]")
test_dataset = load_dataset("tau/commonsense_qa", split="validation")

print(f'Train: {len(train_dataset)}, Validation: {len(validation_dataset)}, Test: {len(test_dataset)}')

## Custom Dataset

In [ ]:
# Custom dataset for CommonsenseQA

# TODO: WARNING: temporarely using dataset from .py file to enable multiple workers when dataloading
from dataset import CommonsenseQADataset, OptimizedCommonsenseQADataset

# Model

## Randomly Initialized Transformer

In [ ]:
# Use the same configuration as RoBERTa-base
config = RobertaConfig(
    vocab_size=50265,
    max_position_embeddings=514,
    hidden_size=768,
    num_hidden_layers=12,
    num_attention_heads=12,
    intermediate_size=3072,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    type_vocab_size=1
)
    
# Create model with random weights
random_transformer = RobertaForMultipleChoice(config)

## Pretrained Transformer

In [ ]:
pretrained_transformer = RobertaForMultipleChoice.from_pretrained("roberta-base")

# Training

## Transformer Training Function

In [ ]:
def train_model(model, train_dataloader, val_dataloader, val_data, model_name, 
                num_epochs=10, learning_rate=2e-5, weight_decay=0.01, 
                warmup_ratio=0.1, gradient_accumulation_steps=4):
    
    # Initialize wandb
    wandb_run = initialize_wandb_run(
        run_name=f'{model_name}_{time.strftime("%Y%m%d-%H%M%S")}',
        learning_rate=learning_rate,
        batch_size=train_dataloader.batch_size if hasattr(train_dataloader, 'batch_size') else None,
        num_epochs=num_epochs,
        model_name=model_name,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        gradient_accumulation_steps=gradient_accumulation_steps
    )
    
    model.to(device)
    
    # Optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # Account for gradient accumulation in total steps calculation
    total_steps = len(train_dataloader) * num_epochs // gradient_accumulation_steps
    
    scheduler = get_linear_schedule_with_warmup(
        optimizer, 
        num_warmup_steps=int(warmup_ratio * total_steps), 
        num_training_steps=total_steps
    )
    
    best_accuracy = 0
    training_stats = []
    
    # Set up wandb to watch the model
    wandb.watch(model, log="all", log_freq=50)
    
    for epoch in range(num_epochs):
        print(f"\n{'='*20} Epoch {epoch+1}/{num_epochs} {'='*20}")
        
        # Training
        model.train()
        total_train_loss = 0
        train_steps = 0
        optimizer.zero_grad()
        
        for batch_idx, batch in enumerate(tqdm(train_dataloader, desc="Training")):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss / gradient_accumulation_steps  # Scale the loss
            total_train_loss += loss.item() * gradient_accumulation_steps  # Multiply by accumulation steps to get the actual loss
            
            # Backward pass
            loss.backward()
            
            # Only update weights after accumulating gradients for several steps
            if (batch_idx + 1) % gradient_accumulation_steps == 0 or (batch_idx + 1 == len(train_dataloader)):
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                
                # Update parameters
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                train_steps += 1
                
                # Log less frequently to reduce overhead (every 10 accumulation steps)
                if train_steps % 10 == 0:
                    wandb.log({
                        "batch_train_loss": total_train_loss / train_steps,
                        "learning_rate": scheduler.get_last_lr()[0]
                    })
        
        avg_train_loss = total_train_loss / (train_steps * gradient_accumulation_steps)
        print(f"Average training loss: {avg_train_loss:.4f}")
        
        # Validation (run every epoch, but could be modified to run less frequently)
        model.eval()
        correct_predictions = 0
        total_eval_loss = 0
        eval_steps = 0
        
        for batch in tqdm(val_dataloader, desc="Validation"):
            with torch.no_grad():
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
                
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                
                loss = outputs.loss
                total_eval_loss += loss.item()
                
                # Get predictions
                logits = outputs.logits
                predictions = torch.argmax(logits, dim=1)
                correct_predictions += (predictions == labels).sum().item()
                
                eval_steps += 1
        
        avg_val_loss = total_eval_loss / eval_steps
        accuracy = correct_predictions / len(val_data)
        
        print(f"Validation Loss: {avg_val_loss:.4f}")
        print(f"Accuracy: {accuracy:.4f}")
        
        # Log epoch metrics
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "accuracy": accuracy,
            "learning_rate": scheduler.get_last_lr()[0]
        })
        
        # Save best model (but only save the state_dict to reduce overhead)
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            
            # Save model to wandb (only state_dict)
            model_path = f"{model_name}_best_model.pt"
            torch.save(model.state_dict(), model_path)
            wandb.save(model_path)
            
            print(f"Best model saved with accuracy: {best_accuracy:.4f}")
            
            # Log best model metrics
            wandb.run.summary["best_accuracy"] = best_accuracy
            wandb.run.summary["best_epoch"] = epoch + 1
        
        # Record stats
        training_stats.append({
            'epoch': epoch + 1,
            'training_loss': avg_train_loss,
            'valid_loss': avg_val_loss,
            'valid_accuracy': accuracy,
        })
    
    print(f"\nTraining complete! Best accuracy: {best_accuracy:.4f}")
    
    # Finish the wandb run
    wandb.finish()
    
    return training_stats

In [ ]:
# Create datasets
train_data = OptimizedCommonsenseQADataset(train_dataset, tokenizer)
val_data = OptimizedCommonsenseQADataset(validation_dataset, tokenizer)

# Create dataloaders
batch_size = 32
train_dataloader = DataLoader(
    train_data, 
    batch_size=batch_size, 
    shuffle=True,
    num_workers=8,
    pin_memory=True # may not affect mps performance but still useful for cuda
)
val_dataloader = DataLoader(
    val_data, 
    batch_size=batch_size,
    num_workers=8,
    pin_memory=True # may not affect mps performance but still useful for cuda
)

In [ ]:
print("\nTraining randomly initialized RoBERTa")
random_stats = train_model(random_transformer, train_dataloader, val_dataloader, val_data, "random_RoBERTa")

print(f"Random RoBERTa best accuracy: {max([stat['valid_accuracy'] for stat in random_stats]):.4f}")

In [ ]:
print("\nTraining pretrained RoBERTa")
pretrained_stats = train_model(pretrained_transformer, train_dataloader, val_dataloader, val_data, "pretrained_RoBERTa")

print(f"Pretrained RoBERTa best accuracy: {max([stat['valid_accuracy'] for stat in pretrained_stats]):.4f}")
    